## **Dependencies**

In [1]:
!pip install -q -U rank-bm25 nltk tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 42.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 5.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.


In [2]:
#!pip install -q -U transformers accelerate torch
#!pip install -q gradio

## **Imports**

In [3]:
import json
import os
import pickle
import re
from typing import Dict, List, Set, Any

import math
import numpy as np
from tqdm import tqdm
from collections import Counter

import torch
from sentence_transformers import CrossEncoder
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModelForSequenceClassification
from rank_bm25 import BM25Okapi

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

In [4]:
nltk.pathsec.ALLOW_PROXIED_FETCH = True
nltk.download('stopwords', quiet=True)

True

In [5]:
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("my_hf_token")
login(token=hf_token)

## **Configuration Paths**

In [6]:
DATA_DIR = "/kaggle/input/datasets/anarvaaa/original-scifact-data"
DEV_CLAIMS_PATH = os.path.join(DATA_DIR, "claims_dev.jsonl")
INDEX_PKL_PATH = os.path.join(DATA_DIR, "scifact_bm25_index.pkl")
BIOBERT_MODEL_PATH = "/kaggle/input/datasets/anarvaaa/biobert-model-files"
MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"

In [7]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using compute device: {DEVICE}")
EXPANSION_MODEL_ID = "HuggingFaceTB/SmolLM2-360M-Instruct"
RERANKER_MODEL_NAME = "BAAI/bge-reranker-base"
CANDIDATE_POOL_SIZE = 50 
FINAL_TOP_K = 5

Using compute device: cuda


## **Preprocessing**

In [8]:
stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()

def tokenize(text: str, remove_stopwords: bool = True, use_stemming: bool = True) -> List[str]:
    # Clean non-alphanumeric noise to protect BM25 token matches
    tokens = re.findall(r"\b[a-zA-Z0-9]+(?:-[a-zA-Z0-9]+)*\b", text.lower())
    if remove_stopwords:
        tokens = [t for t in tokens if t not in stop_words]
    if use_stemming:
        tokens = [stemmer.stem(t) for t in tokens]
    return tokens

## **Load Index**

In [9]:
print(f"Loading BM25 index from: {INDEX_PKL_PATH}")
with open(INDEX_PKL_PATH, "rb") as f:
    bm25_artifacts = pickle.load(f)

bm25: BM25Okapi = bm25_artifacts["bm25_model"]
doc_ids: List[int] = bm25_artifacts["doc_ids"]
doc_metadata: Dict[int, Dict[str, Any]] = bm25_artifacts["doc_metadata"]

print(f"Successfully loaded index with {len(doc_ids):,} indexed documents.")

Loading BM25 index from: /kaggle/input/datasets/anarvaaa/original-scifact-data/scifact_bm25_index.pkl
Successfully loaded index with 5,183 indexed documents.


## **Load Models**

In [10]:
print("Loading trained BioBERT for Verdict")

biobert_tokenizer = AutoTokenizer.from_pretrained(
    BIOBERT_MODEL_PATH
)

biobert_model = AutoModelForSequenceClassification.from_pretrained(
    BIOBERT_MODEL_PATH
)

biobert_model.to(DEVICE)
biobert_model.eval()

print("BioBERT loaded successfully.")
print(biobert_model.config.id2label)

Loading trained BioBERT for Verdict


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BioBERT loaded successfully.
{0: 'CONTRADICT', 1: 'SUPPORT', 2: 'NEI'}


In [11]:
print(f"Loading Cross-Encoder Reranker: {RERANKER_MODEL_NAME} for Re-ranking")
reranker = CrossEncoder(RERANKER_MODEL_NAME, max_length=512, device=DEVICE)

Loading Cross-Encoder Reranker: BAAI/bge-reranker-base for Re-ranking


config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

In [12]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=hf_token)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=hf_token,
    dtype=torch.float16,      # NOT torch_dtype (deprecated, as you already found)
    device_map="auto",
)
model.eval()

DEVICE = next(model.parameters()).device
NUM_LAYERS = model.config.num_hidden_layers
HIDDEN_DIM = model.config.hidden_size
print(f"Loaded {MODEL_NAME}: {NUM_LAYERS} layers, hidden_dim={HIDDEN_DIM}, device={DEVICE}")

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Loaded meta-llama/Llama-3.2-3B-Instruct: 28 layers, hidden_dim=3072, device=cuda:0


## **Helper Functions**

In [13]:
def retrieve_bm25(query_str: str, k: int = CANDIDATE_POOL_SIZE) -> List[Dict[str, Any]]:
    tokens = tokenize(query_str)
    scores = bm25.get_scores(tokens)
    top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
    
    results = []
    for rank, idx in enumerate(top_indices, start=1):
        d_id = doc_ids[idx]
        results.append({
            "doc_id": d_id,
            "rank": rank,
            "score": round(float(scores[idx]), 4),
            "title": doc_metadata[d_id]["title"],
            "abstract_text": doc_metadata[d_id]["abstract_text"]
        })
    return results

In [14]:
def rerank_documents_hybrid(
    expanded_query: str, 
    candidate_docs: List[Dict[str, Any]], 
    alpha: float = 0.7
) -> List[Dict[str, Any]]:
    """
    Hybrid Re-ranking: 
        Combines Min-Max normalized Cross-Encoder scores with BM25 scores to prevent query drift.
    """
    if not candidate_docs:
        return []

    # 1. Match against Expanded Query
    pairs = [
        [expanded_query, f"{doc['title']} {doc['abstract_text']}".strip()] 
        for doc in candidate_docs
    ]
    
    ce_scores = reranker.predict(pairs)

    # 2. Normalize BM25 and Cross-Encoder scores for linear interpolation
    bm25_raw = [d["score"] for d in candidate_docs]
    ce_raw = list(ce_scores)

    min_bm25, max_bm25 = min(bm25_raw), max(bm25_raw)
    min_ce, max_ce = min(ce_raw), max(ce_raw)

    bm25_norm = [(s - min_bm25) / (max_bm25 - min_bm25 + 1e-6) for s in bm25_raw]
    ce_norm = [(s - min_ce) / (max_ce - min_ce + 1e-6) for s in ce_raw]

    # 3. Score Fusion: Final_Score = alpha * CE_norm + (1 - alpha) * BM25_norm
    reranked_docs = []
    for idx, doc in enumerate(candidate_docs):
        doc_copy = doc.copy()
        fusion_score = (alpha * ce_norm[idx]) + ((1 - alpha) * bm25_norm[idx])
        doc_copy["ce_score"] = float(ce_scores[idx])
        doc_copy["fusion_score"] = float(fusion_score)
        reranked_docs.append(doc_copy)

    return sorted(reranked_docs, key=lambda x: x["fusion_score"], reverse=True)

In [15]:
def predict_verdict(claim: str, document_text: str) -> dict:
    # 1. Match exact training sequence format: Single concatenated string
    combined_text = f"{claim} {document_text}"

    # 2. Tokenize once with single-string truncation
    inputs = biobert_tokenizer(
        combined_text,
        truncation=True,
        padding=True,
        max_length=512,
        return_tensors="pt"
    ).to(DEVICE)

    with torch.no_grad():
        outputs = biobert_model(**inputs)

    # 3. Compute Softmax across all 3 classes [CONTRADICT, SUPPORT, NEI]
    probabilities = torch.softmax(outputs.logits, dim=-1)[0]
    predicted_id = torch.argmax(probabilities).item()

    return {
        "verdict": biobert_model.config.id2label[predicted_id],
        "contradict_probability": float(probabilities[0]),
        "support_probability": float(probabilities[1]),
        "nei_probability": float(probabilities[2])
    }

### Verdict Aggregation

Before the LLM ever sees the documents, collapse the 5 per-document BioBERT verdicts into a single overall verdict by majority vote. This becomes the ground truth the LLM must explain — it is no longer allowed to independently re-decide SUPPORT/CONTRADICT/MIXED from the raw abstracts.


In [16]:
def aggregate_verdict(top_5_documents: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    Aggregate the 5 per-document BioBERT verdicts (SUPPORT / CONTRADICT / NEI)
    into a single overall verdict with 4 confidence tiers:

      STRONG    -> one class (SUPPORT or CONTRADICT) has 4 or 5 of the 5
                   votes, regardless of what the remaining vote(s) are.

      MODERATE  -> exactly 3 votes for one class, and the other 2 are BOTH
                   NEI (i.e. nothing actually opposes the majority -- the
                   rest is just inconclusive). Reported as a plain,
                   confident SUPPORT / CONTRADICT.

      MILD      -> exactly 3 votes for one class, with the remaining 2
                   split as 1 opposing vote + 1 NEI. Reported as a
                   "mild" SUPPORT / CONTRADICT, with the dissenting /
                   uncertain document(s) flagged for the LLM to mention.

      MIXED     -> everything else: 4 or 5 NEI votes, 3 NEI votes (any
                   split of the remaining 2), a direct 3-2 clash between
                   SUPPORT and CONTRADICT, or a near-even split like
                   2-2-1. No reliable majority exists, so this is
                   reported as a MIXED SIGNAL rather than forcing a
                   direction.
    """
    total = len(top_5_documents)

    support_votes = sum(1 for d in top_5_documents if d["verdict"] == "SUPPORT")
    contradict_votes = sum(1 for d in top_5_documents if d["verdict"] == "CONTRADICT")
    nei_votes = sum(1 for d in top_5_documents if d["verdict"] == "NEI")

    avg_support_prob = sum(d["support_probability"] for d in top_5_documents) / total
    avg_contradict_prob = sum(d["contradict_probability"] for d in top_5_documents) / total

    def others(majority_class):
        return [d for d in top_5_documents if d["verdict"] != majority_class]

    if support_votes >= 4:
        confidence, overall_verdict = "STRONG", "SUPPORT"
        dissenting_docs = others("SUPPORT")

    elif contradict_votes >= 4:
        confidence, overall_verdict = "STRONG", "CONTRADICT"
        dissenting_docs = others("CONTRADICT")

    elif nei_votes >= 4:
        confidence, overall_verdict = "MIXED", "MIXED SIGNAL"
        dissenting_docs = []

    elif support_votes == 3 and contradict_votes == 0 and nei_votes == 2:
        confidence, overall_verdict = "MODERATE", "SUPPORT"
        dissenting_docs = others("SUPPORT")

    elif support_votes == 3 and contradict_votes == 1 and nei_votes == 1:
        confidence, overall_verdict = "MILD", "SUPPORT"
        dissenting_docs = others("SUPPORT")

    elif contradict_votes == 3 and support_votes == 0 and nei_votes == 2:
        confidence, overall_verdict = "MODERATE", "CONTRADICT"
        dissenting_docs = others("CONTRADICT")

    elif contradict_votes == 3 and support_votes == 1 and nei_votes == 1:
        confidence, overall_verdict = "MILD", "CONTRADICT"
        dissenting_docs = others("CONTRADICT")

    else:
        # remaining cases: NEI == 3 (any split), a direct 3-2 SUPPORT/CONTRADICT
        # clash, or a near-even split such as 2-2-1 -- no reliable majority
        confidence, overall_verdict = "MIXED", "MIXED SIGNAL"
        dissenting_docs = []

    return {
        "overall_verdict": overall_verdict,
        "confidence": confidence,
        "support_votes": support_votes,
        "contradict_votes": contradict_votes,
        "nei_votes": nei_votes,
        "total_documents": total,
        "avg_support_probability": avg_support_prob,
        "avg_contradict_probability": avg_contradict_prob,
        "dissenting_doc_ranks": [d["rank"] for d in dissenting_docs],
    }


In [17]:
def build_summary_prompt(query, top_5_documents, aggregate):

    evidence_blocks = []

    for doc in top_5_documents:

        block = f"""
DOCUMENT {doc['rank']}
Document ID: {doc['doc_id']}
Title: {doc['title']}

BioBERT verdict: {doc['verdict']}
Support probability: {doc['support_probability']:.4f}
Contradict probability: {doc['contradict_probability']:.4f}

ABSTRACT:
{doc['abstract_text']}
"""

        evidence_blocks.append(block)

    evidence = "\n".join(evidence_blocks)

    confidence = aggregate["confidence"]
    overall_verdict = aggregate["overall_verdict"]
    vote_summary = (
        f"{aggregate['support_votes']} of {aggregate['total_documents']} SUPPORT, "
        f"{aggregate['contradict_votes']} of {aggregate['total_documents']} CONTRADICT, "
        f"{aggregate['nei_votes']} of {aggregate['total_documents']} inconclusive (NEI)"
    )

    # The directive tells the LLM exactly what conclusion it must land on,
    # and how firmly it is allowed to state it, so it can't independently
    # re-decide "mixed" when the aggregation found a clear majority, and
    # can't force a confident verdict when the aggregation found none.
    if confidence == "STRONG":
        directive = f"""
OVERALL VERDICT (already decided, do not override): {overall_verdict}
Vote breakdown: {vote_summary}.

This is a STRONG, near-unanimous verdict. State it directly and
confidently. Do not call the evidence "mixed".
"""
    elif confidence == "MODERATE":
        directive = f"""
OVERALL VERDICT (already decided, do not override): {overall_verdict}
Vote breakdown: {vote_summary}.

A clear majority (3 of 5) point to {overall_verdict}, and the remaining
2 documents are inconclusive (NEI) rather than actually opposing it --
nothing in the evidence contradicts this verdict. State it directly and
confidently as {overall_verdict}. You may briefly note that 2 documents
did not provide clear evidence either way, but do NOT soften this into
"mild" language and do NOT call the evidence "mixed".
"""
    elif confidence == "MILD":
        dissent = ", ".join(f"Document {r}" for r in aggregate["dissenting_doc_ranks"])
        directive = f"""
OVERALL VERDICT (already decided, do not override): MILD {overall_verdict}
Vote breakdown: {vote_summary}.

A majority (3 of 5) point to {overall_verdict}, but one document
directly disagrees and one is inconclusive. Explicitly acknowledge the
dissenting/inconclusive document(s) ({dissent}) and note that they
temper your confidence, but your bottom-line conclusion must still be
{overall_verdict} -- phrase it as "the evidence leans toward
{overall_verdict.lower()}" or "mild {overall_verdict.lower()}", not as
a flat, unqualified verdict.
"""
    else:  # MIXED
        directive = f"""
OVERALL VERDICT (already decided, do not override): MIXED SIGNAL
Vote breakdown: {vote_summary}.

There is NO reliable majority among the 5 documents -- the evidence is
genuinely mixed or inconclusive. Do not force a SUPPORT or CONTRADICT
conclusion. Instead, clearly state that the evidence is mixed/
inconclusive, and briefly summarize which documents point which way.
"""

    prompt = f"""
You are a scientific evidence summarization assistant.

A BioBERT classifier has already independently scored each of the five
retrieved documents below as SUPPORT, CONTRADICT, or NEI (not enough
information) with respect to the claim. Those five per-document
verdicts have already been aggregated into a single OVERALL VERDICT.
This OVERALL VERDICT is final and has already been decided -- it is
NOT your job to re-decide it, only to explain it.

CLAIM:
{query}
{directive}
{evidence}

Your ONLY task is to explain, using the abstracts above, why the
evidence leads to the OVERALL VERDICT stated above.

Instructions:

1. Follow the OVERALL VERDICT and its phrasing guidance exactly as
   given above.
2. Explain, using only the provided abstracts, why the evidence
   leads to that verdict.
3. Mention the relevant document numbers when describing evidence,
   e.g. [Document 1].
4. Do NOT introduce facts that are not present in the provided
   abstracts.
5. Do not treat the BioBERT verdict as evidence itself; the abstracts
   are the evidence -- use them to justify the verdict, not to
   re-derive it.
6. Keep the explanation concise (roughly 4-6 sentences).

Return only the grounded explanation.
"""

    return prompt


In [18]:
def generate_grounded_summary(query, top_5_documents, aggregate):

    prompt = build_summary_prompt(
        query,
        top_5_documents,
        aggregate
    )

    messages = [
        {
            "role": "system",
            "content": (
                "You are a scientific evidence summarization "
                "assistant. Your role is to EXPLAIN a verdict that "
                "has already been determined by an upstream "
                "classifier -- not to independently judge the claim "
                "yourself. Be strictly grounded in the provided "
                "documents."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt",
        truncation=True,
        max_length=16000
    )

    inputs = {
        k: v.to(model.device)
        for k, v in inputs.items()
    }

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=400,
            do_sample=False,
            temperature=None,
            top_p=None
        )

    generated_tokens = outputs[
        0
    ][inputs["input_ids"].shape[-1]:]

    summary = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return summary.strip()


In [19]:
def retrieve_rerank_verdict_summarize(query):

    # ----------------------------------------
    # 1. BM25
    # ----------------------------------------
    bm25_docs = retrieve_bm25(
        query,
        k=CANDIDATE_POOL_SIZE
    )

    # ----------------------------------------
    # 2. Cross-encoder reranking
    # ----------------------------------------
    reranked_docs = rerank_documents_hybrid(
        query,
        bm25_docs,
        alpha=0.7
    )

    # ----------------------------------------
    # 3. Top 5
    # ----------------------------------------
    top_5_docs = reranked_docs[:FINAL_TOP_K]

    # ----------------------------------------
    # 4. BioBERT verdict (per document)
    # ----------------------------------------
    final_results = []

    for rank, doc in enumerate(top_5_docs, start=1):

        verdict_result = predict_verdict(
            query,
            doc["abstract_text"]
        )

        final_results.append({
            "rank": rank,
            "doc_id": doc["doc_id"],
            "title": doc["title"],
            "abstract_text": doc["abstract_text"],

            "bm25_score": doc["score"],
            "ce_score": doc["ce_score"],
            "fusion_score": doc["fusion_score"],

            "verdict": verdict_result["verdict"],
            "contradict_probability":
                verdict_result["contradict_probability"],
            "support_probability":
                verdict_result["support_probability"],
            "nei_probability":
                verdict_result["nei_probability"]
        })

    # ----------------------------------------
    # 5. Aggregate the 5 verdicts into ONE overall verdict
    #    (this is now the fixed conclusion, decided before the LLM runs)
    # ----------------------------------------
    aggregate = aggregate_verdict(final_results)

    # ----------------------------------------
    # 6. Llama grounded explanation of the aggregated verdict
    # ----------------------------------------
    summary = generate_grounded_summary(
        query,
        final_results,
        aggregate
    )

    return {
        "query": query,
        "top_5_documents": final_results,
        "aggregate_verdict": aggregate,
        "grounded_summary": summary
    }


## **Test Run**

In [42]:
## **Test Run**

# 141 -> 5 support, 0 contradict, 0 NEI
# 598 -> 0 support, 5 contradict, 0 NEI
# 1270-> 1 support, 4 contradict, 0 NEI
# 1187-> 4 support, 0 contradict, 1 NEI
# 781 -> 3 support, 0 contradict, 2 NEI
# 775 -> 2 support, 0 contradict, 3 NEI
# 54  -> 1 support, 2 contradict, 2 NEI
# 1163-> 2 support, 2 contradict, 1 NEI

TEST_CLAIM_IDS = [141, 598, 1163, 781]


def load_claims_by_ids(claim_ids, claims_path=DEV_CLAIMS_PATH):
    """
    Load specific claims from claims_dev.jsonl using their IDs.
    """
    
    claim_ids = set(claim_ids)
    selected_claims = {}

    with open(claims_path, "r", encoding="utf-8") as f:
        for line in f:
            claim = json.loads(line)

            if claim["id"] in claim_ids:
                selected_claims[claim["id"]] = claim["claim"]

    missing_ids = claim_ids - set(selected_claims.keys())

    if missing_ids:
        raise ValueError(
            f"The following claim IDs were not found in {claims_path}: "
            f"{sorted(missing_ids)}"
        )

    return selected_claims


def run_test_claims(
    claim_ids=TEST_CLAIM_IDS,
    claims_path=DEV_CLAIMS_PATH
):
    """
    Extract the specified claims from claims_dev.jsonl
    and run the complete retrieval -> reranking -> BioBERT
    verdict -> Llama summarization pipeline on each claim.
    """

    selected_claims = load_claims_by_ids(
        claim_ids,
        claims_path
    )

    results = {}

    for claim_id in claim_ids:

        query = selected_claims[claim_id]
        #print(query)

        result = retrieve_rerank_verdict_summarize(
            query
        )

        result["claim_id"] = claim_id

        results[claim_id] = result

    return results

## **Generation**

In [43]:
results = run_test_claims()


for claim_id in TEST_CLAIM_IDS:

    result = results[claim_id]

    print("\n" + "=" * 100)
    print(f"CLAIM ID: {claim_id}")
    print("=" * 100)

    print("\nQUERY")
    print("-" * 100)
    print(result["query"])

    print("\n" + "=" * 100)
    print("TOP 5 DOCUMENTS + BIOBERT VERDICTS")
    print("=" * 100)

    for doc in result["top_5_documents"]:

        print(f"\nDocument {doc['rank']}")
        print(f"ID: {doc['doc_id']}")
        print(f"Title: {doc['title']}")
        print(f"Verdict: {doc['verdict']}")

        print(
            f"Support probability: "
            f"{doc['support_probability']:.4f}"
        )

        print(
            f"Contradict probability: "
            f"{doc['contradict_probability']:.4f}"
        )

    print("\n" + "=" * 100)
    print("GROUNDED SUMMARY")
    print("=" * 100)

    print(result["grounded_summary"])

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



CLAIM ID: 141

QUERY
----------------------------------------------------------------------------------------------------
Auditory entrainment is strengthened when people see congruent visual and auditory information.

TOP 5 DOCUMENTS + BIOBERT VERDICTS

Document 1
ID: 14437255
Title: Congruent Visual Speech Enhances Cortical Entrainment to Continuous Auditory Speech in Noise-Free Conditions.
Verdict: SUPPORT
Support probability: 0.6671
Contradict probability: 0.2007

Document 2
ID: 10624000
Title: Lip movements entrain the observers’ low-frequency brain oscillations to facilitate speech intelligibility
Verdict: SUPPORT
Support probability: 0.6602
Contradict probability: 0.2143

Document 3
ID: 6955746
Title: Auditory Cortex Tracks Both Auditory and Visual Stimulus Dynamics Using Low-Frequency Neuronal Phase Modulation
Verdict: SUPPORT
Support probability: 0.5667
Contradict probability: 0.2013

Document 4
ID: 38794814
Title: Frequency modulation entrains slow neural oscillations and op

## **Gradio Demo**

Simple dropdown UI over the pipeline defined above. Run this after all the model-loading cells above have executed (the demo reuses `bm25`, `reranker`, `biobert_model`, `model`, `tokenizer`, etc. already in memory — no reloading).


In [44]:
import gradio as gr

# The demo claims (SciFact dev set)
CLAIM_CHOICES = [
    "Auditory entrainment is strengthened when people see congruent visual and auditory information.",
    "Incidence rates of cervical cancer have increased due to nationwide screening programs based primarily on cytology to detect uterine cervical cancer.",
    "The DdrB protein from Deinococcus radiodurans is an alternative SSB.",
    "Mice that lack Interferon-γ or its receptor exhibit high resistance to experimental autoimmune myocarditis."
]

_TIER_EMOJI = {
    ("STRONG", "SUPPORT"): "✅",
    ("STRONG", "CONTRADICT"): "❌",
    ("MODERATE", "SUPPORT"): "✅",
    ("MODERATE", "CONTRADICT"): "❌",
    ("MILD", "SUPPORT"): "🟢",
    ("MILD", "CONTRADICT"): "🟠",
    ("MIXED", "MIXED SIGNAL"): "⚖️",
}

_DOC_EMOJI = {"SUPPORT": "✅", "CONTRADICT": "❌", "NEI": "❔"}


def format_result_markdown(result: dict) -> str:
    md = []
    md.append(f"### 🧾 Claim\n{result['query']}\n")

    # ---- Overall (aggregated) verdict banner ----
    agg = result["aggregate_verdict"]
    banner_emoji = _TIER_EMOJI.get((agg["confidence"], agg["overall_verdict"]), "❓")

    if agg["confidence"] == "STRONG":
        headline_text = agg["overall_verdict"]
        tier_note = "strong, near-unanimous"
    elif agg["confidence"] == "MODERATE":
        headline_text = agg["overall_verdict"]
        tier_note = "moderate — rest inconclusive, nothing opposing"
    elif agg["confidence"] == "MILD":
        headline_text = f"Mild {agg['overall_verdict']}"
        tier_note = "mild — one document dissents"
    else:
        headline_text = "Mixed Signal"
        tier_note = "no reliable majority"

    md.append(
        f"## {banner_emoji} Overall Verdict: **{headline_text}**  _({tier_note})_\n"
        f"_{agg['support_votes']} SUPPORT · {agg['contradict_votes']} CONTRADICT · "
        f"{agg['nei_votes']} inconclusive (NEI), out of {agg['total_documents']} documents_\n"
    )

    md.append("---\n### 📚 Top 5 Retrieved & Reranked Documents\n")

    for doc in result["top_5_documents"]:
        verdict = doc["verdict"]
        emoji = _DOC_EMOJI.get(verdict, "❓")

        md.append(f"**{doc['rank']}. {doc['title']}**  _(doc_id: {doc['doc_id']})_\n")
        md.append(f"- Verdict: {emoji} **{verdict}**")
        md.append(
            f"- Support: `{doc['support_probability']:.3f}`  |  "
            f"Contradict: `{doc['contradict_probability']:.3f}`  |  "
            f"NEI: `{doc['nei_probability']:.3f}`"
        )
        md.append(
            f"- Fusion score: `{doc['fusion_score']:.3f}` "
            f"(BM25: `{doc['bm25_score']:.3f}`, Cross-Encoder: `{doc['ce_score']:.3f}`)"
        )
        md.append(
            f"<details><summary>Show abstract</summary>\n\n{doc['abstract_text']}\n\n</details>\n"
        )

    md.append(f"---\n### 🤖 Explanation of the {headline_text} Verdict (LLM-generated)\n")
    md.append(result["grounded_summary"])

    return "\n".join(md)


def run_demo(selected_claim: str) -> str:
    if not selected_claim:
        return "Please select a claim from the dropdown."
    result = retrieve_rerank_verdict_summarize(selected_claim)
    return format_result_markdown(result)


with gr.Blocks(title="SciVerify — Scientific Claim Verification") as demo:
    gr.Markdown(
        "# 🔬 SciVerify\n"
        "Select a scientific claim below. The pipeline retrieves evidence "
        "with BM25 + cross-encoder reranking, classifies each document's "
        "stance with BioBERT (SUPPORT / CONTRADICT / NEI), aggregates those "
        "5 verdicts into a tiered overall verdict (Strong / Mild / Mixed "
        "Signal), and then asks the LLM to explain that verdict rather "
        "than re-decide it."
    )

    claim_dropdown = gr.Dropdown(
        choices=CLAIM_CHOICES,
        label="Select a claim",
        value=CLAIM_CHOICES[0],
    )
    run_btn = gr.Button("Verify Claim", variant="primary")
    output_md = gr.Markdown()

    run_btn.click(fn=run_demo, inputs=claim_dropdown, outputs=output_md)

demo.launch(share=True, debug=True)


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://c8fa6da4b1f29573b1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://c8fa6da4b1f29573b1.gradio.live
